In [1]:
import numpy as np
import plotly.graph_objects as go
import os

# ---- Config ----
scan_index = 1  # change to load planck_1.npy, planck_2.npy, etc.

npy_dir = r"planck_scans/npy"
scaled_path = r"experiments/geotransformer.faces.stage4.gse.k3.max.oacl.stage2.sinkhorn/plank_scaled.npy"

# ---- Load ----
src_points = np.load(os.path.join(npy_dir, f"planck_{scan_index}.npy")).astype(np.float64)
scaled_points = np.load(scaled_path)

# ---- Plot ----
all_points = np.vstack([src_points, scaled_points])
center = (all_points.min(0) + all_points.max(0)) / 2
extent = (all_points.max(0) - all_points.min(0)).max() / 2

fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=src_points[:, 0], y=src_points[:, 1], z=src_points[:, 2],
    mode='markers', marker=dict(size=2, color='steelblue'),
    name=f"planck_{scan_index} (raw)"
))

fig.add_trace(go.Scatter3d(
    x=scaled_points[:, 0], y=scaled_points[:, 1], z=scaled_points[:, 2],
    mode='markers', marker=dict(size=2, color='tomato'),
    name="plank_scaled"
))

fig.update_layout(
    scene=dict(
        xaxis=dict(range=[center[0]-extent, center[0]+extent]),
        yaxis=dict(range=[center[1]-extent, center[1]+extent]),
        zaxis=dict(range=[center[2]-extent, center[2]+extent]),
        aspectmode='cube'
    )
)

fig.show()

In [2]:
import numpy as np
import os

npy_dir = r"planck_scans/npy"
scaled_path = r"experiments/geotransformer.faces.stage4.gse.k3.max.oacl.stage2.sinkhorn/plank_scaled.npy"

scaled = np.load(scaled_path)
scaled_span = scaled.max(0) - scaled.min(0)
scaled_max_span = scaled_span.max()

files = sorted([f for f in os.listdir(npy_dir) if f.endswith('.npy')])

print(f"{'Name':<22} {'N pts':>8}  {'span_x':>8} {'span_y':>8} {'span_z':>8}  {'scale_factor':>12}")
print("-" * 80)
for fname in files:
    pts = np.load(os.path.join(npy_dir, fname))
    span = pts.max(0) - pts.min(0)
    factor = scaled_max_span / span.max()
    print(f"{fname:<22} {len(pts):>8}  {span[0]:>8.3f} {span[1]:>8.3f} {span[2]:>8.3f}  {factor:>12.5f}")

print("-" * 80)
print(f"{'plank_scaled (ref)':<22} {len(scaled):>8}  {scaled_span[0]:>8.3f} {scaled_span[1]:>8.3f} {scaled_span[2]:>8.3f}  {'1.00000':>12}")
print()
print("scale_factor = multiply planck_scans/npy points by this to match plank_scaled max span")


Name                      N pts    span_x   span_y   span_z  scale_factor
--------------------------------------------------------------------------------
planck_1.npy              29588   153.116  186.665   94.093       0.00849
planck_10.npy             33961   152.399  195.020   96.279       0.00813
planck_11.npy             27008   143.228  185.209   87.378       0.00856
planck_12.npy             25202   142.935  166.792   84.228       0.00951
planck_13.npy             35100   152.576  194.882   99.447       0.00814
planck_14.npy             28395   143.929  179.283   89.511       0.00884
planck_15.npy             28719   141.699  183.028   95.624       0.00866
planck_16.npy             36014   162.859  207.600   96.429       0.00764
planck_17.npy             32197   147.337  188.904  103.658       0.00839
planck_18.npy             36419   146.973  200.496  103.055       0.00791
planck_19.npy             30308   135.613  183.963   96.035       0.00862
planck_2.npy              25949

In [3]:
import numpy as np
import os

npy_dir = r"planck_scans/npy"
out_dir = r"planck_scans/npy_scaled"
scaled_path = r"experiments/geotransformer.faces.stage4.gse.k3.max.oacl.stage2.sinkhorn/plank_scaled.npy"
target_n = 2000
rng = np.random.default_rng(42)

os.makedirs(out_dir, exist_ok=True)

scaled_ref = np.load(scaled_path)
ref_max_span = (scaled_ref.max(0) - scaled_ref.min(0)).max()

files = sorted([f for f in os.listdir(npy_dir) if f.endswith('.npy')])

for fname in files:
    pts = np.load(os.path.join(npy_dir, fname)).astype(np.float64)
    factor = ref_max_span / (pts.max(0) - pts.min(0)).max()
    pts = pts * factor
    pts = pts - pts.mean(0)
    idx = rng.choice(len(pts), size=target_n, replace=False)
    pts = pts[idx]
    out_path = os.path.join(out_dir, fname)
    np.save(out_path, pts.astype(np.float32))
    print(f"{fname}: scale={factor:.5f}  ->  {pts.shape}  centroid={pts.mean(0).round(4)}")

print(f"\nSaved {len(files)} files to {out_dir}")


planck_1.npy: scale=0.00849  ->  (2000, 3)  centroid=[-0.006   0.0011 -0.0033]
planck_10.npy: scale=0.00813  ->  (2000, 3)  centroid=[-0.0047 -0.0093 -0.0075]
planck_11.npy: scale=0.00856  ->  (2000, 3)  centroid=[ 0.004  -0.0069 -0.0014]
planck_12.npy: scale=0.00951  ->  (2000, 3)  centroid=[ 0.0135 -0.0064  0.0021]
planck_13.npy: scale=0.00814  ->  (2000, 3)  centroid=[-0.0087  0.0051  0.0021]
planck_14.npy: scale=0.00884  ->  (2000, 3)  centroid=[ 0.0032 -0.011   0.0007]
planck_15.npy: scale=0.00866  ->  (2000, 3)  centroid=[ 0.0072  0.0017 -0.0034]
planck_16.npy: scale=0.00764  ->  (2000, 3)  centroid=[-0.0022 -0.0161 -0.0098]
planck_17.npy: scale=0.00839  ->  (2000, 3)  centroid=[ 0.0059  0.0065 -0.002 ]
planck_18.npy: scale=0.00791  ->  (2000, 3)  centroid=[-0.0032  0.0025  0.0059]
planck_19.npy: scale=0.00862  ->  (2000, 3)  centroid=[0.0028 0.0014 0.0009]
planck_2.npy: scale=0.00944  ->  (2000, 3)  centroid=[ 0.0114 -0.0026 -0.003 ]
planck_20.npy: scale=0.00900  ->  (2000, 3)  

In [4]:
# ---- Config ----
scan_index = 10  # change to load planck_1.npy, planck_2.npy, etc.

npy_dir = r"planck_scans/npy_scaled"
scaled_path = r"experiments/geotransformer.faces.stage4.gse.k3.max.oacl.stage2.sinkhorn/plank_scaled.npy"

# ---- Load ----
src_points = np.load(os.path.join(npy_dir, f"planck_{scan_index}.npy")).astype(np.float64)
scaled_points = np.load(scaled_path)

# ---- Plot ----
all_points = np.vstack([src_points, scaled_points])
center = (all_points.min(0) + all_points.max(0)) / 2
extent = (all_points.max(0) - all_points.min(0)).max() / 2

fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=src_points[:, 0], y=src_points[:, 1], z=src_points[:, 2],
    mode='markers', marker=dict(size=2, color='steelblue'),
    name=f"planck_{scan_index} (raw)"
))

fig.add_trace(go.Scatter3d(
    x=scaled_points[:, 0], y=scaled_points[:, 1], z=scaled_points[:, 2],
    mode='markers', marker=dict(size=2, color='tomato'),
    name="plank_scaled"
))

fig.update_layout(
    scene=dict(
        xaxis=dict(range=[center[0]-extent, center[0]+extent]),
        yaxis=dict(range=[center[1]-extent, center[1]+extent]),
        zaxis=dict(range=[center[2]-extent, center[2]+extent]),
        aspectmode='cube'
    )
)

fig.show()